# Lesson 15: Course Review and Statistical Reporting

**Course:** hypothesis_testing_and_statistical_inference  
**Prerequisites:** Basic Python and descriptive statistics

This lesson follows a repeatable workflow: define the question, encode the design,
check assumptions, estimate the effect, quantify uncertainty, test the claim, and
communicate both statistical and practical meaning.


## Learning objectives

1. Navigate the complete hypothesis-test selection map
2. Recognize recurring interpretation and reporting mistakes
3. Use a minimum reporting standard for any test family
4. Plan next steps for more advanced inference


## Setup

Run this cell first. Every example uses a fixed random seed so results are reproducible.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Work whether the kernel starts in the course root or a level folder.
working_dir = Path.cwd()
COURSE_ROOT = working_dir if (working_dir / "data").exists() else working_dir.parent
DATA_DIR = COURSE_ROOT / "data"

rng = np.random.default_rng(20260920)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.precision", 4)


## 15.1 Quick selection map

| Outcome and design | Primary family | Common alternative or extension |
|---|---|---|
| One quantitative sample vs target | one-sample t | signed-rank / permutation |
| Two independent quantitative groups | Welch t | Mann–Whitney / permutation |
| Two paired quantitative conditions | paired t | signed-rank |
| 3+ independent quantitative groups | ANOVA / Welch ANOVA | Kruskal–Wallis |
| 3+ repeated quantitative conditions | RM-ANOVA / mixed model | Friedman |
| One proportion vs target | exact binomial / z | — |
| Two independent proportions | two-proportion z / chi-square | Fisher exact |
| Two paired binary outcomes | McNemar | exact McNemar |
| Two categorical variables | chi-square independence | exact/simulation methods |
| Continuous association | Pearson / regression | Spearman / Kendall |
| Time-to-event by group | log-rank | Cox model |
| Clustered/repeated observations | mixed model / GEE | cluster-level analysis |


## 15.2 Minimum reporting standard

Every result should make these visible:

1. Research question, population, outcome, groups, and observational unit.
2. Design and sampling/randomization process.
3. Estimand and hypotheses, including sidedness and alpha.
4. Descriptive statistics and a plot of the raw data or sufficient counts.
5. Assumption checks and any deviations from the analysis plan.
6. Estimate in natural units, confidence interval, and effect size.
7. Test statistic, degrees of freedom when applicable, and exact p-value.
8. Multiplicity method and hypothesis family.
9. Practical interpretation, limitations, and next decision.


In [2]:
def recommend_test(outcome, groups=1, paired=False, expected_sparse=False):
    # A teaching aid—not a substitute for design expertise.
    if outcome == "time-to-event":
        return "Log-rank for curves; survival regression for adjustment"
    if outcome == "categorical":
        if paired:
            return "McNemar (2 conditions) or Cochran Q (3+ conditions)"
        if groups == 1:
            return "Chi-square goodness-of-fit or exact binomial/multinomial"
        return "Fisher exact" if expected_sparse else "Chi-square independence / proportion test"
    if outcome in {"continuous", "ordinal"}:
        if groups == 1:
            return "One-sample t; signed-rank/permutation when appropriate"
        if groups == 2:
            return "Paired t / signed-rank" if paired else "Welch t / Mann-Whitney"
        return "RM-ANOVA/mixed model/Friedman" if paired else "ANOVA/Welch ANOVA/Kruskal-Wallis"
    return "Clarify the outcome, estimand, and dependence structure"

# Load each real course dataset so the review is anchored in the same evidence
# used throughout the earlier lessons.
dataset_rows = {
    "ANES": len(pd.read_csv(DATA_DIR / "anes96_clean.csv")),
    "Grunfeld": len(pd.read_csv(DATA_DIR / "grunfeld_investment.csv")),
    "Spector": len(pd.read_csv(DATA_DIR / "spector_program.csv")),
    "Heart transplant": len(pd.read_csv(DATA_DIR / "heart_transplant_survival.csv")),
}

data_examples = [
    {"dataset": "ANES", "outcome": "continuous", "groups": 2, "paired": False, "sparse": False},
    {"dataset": "Grunfeld", "outcome": "continuous", "groups": 2, "paired": True, "sparse": False},
    {"dataset": "Spector", "outcome": "categorical", "groups": 2, "paired": False, "sparse": True},
    {"dataset": "Heart transplant", "outcome": "time-to-event", "groups": 2, "paired": False, "sparse": False},
]
pd.DataFrame([{**row, "rows": dataset_rows[row["dataset"]],
               "starting point": recommend_test(
                   row["outcome"], row["groups"], row["paired"], row["sparse"])}
              for row in data_examples])


,dataset,outcome,groups,paired,sparse,rows,starting point
0,ANES,continuous,2,False,False,944,Welch t / Mann-Whitney
1,Grunfeld,continuous,2,True,False,220,Paired t / signed-rank
2,Spector,categorical,2,False,True,32,Fisher exact
3,Heart transplant,time-to-event,2,False,False,69,Log-rank for curves; survival regression for a...


## 15.3 Common mistakes to avoid

- Treating p-values as probabilities that a hypothesis is true.
- Declaring "no effect" from a non-significant result.
- Choosing a test only from a normality test p-value.
- Ignoring pairing, clustering, repeated users, or time dependence.
- Reporting standardized effects without raw-unit effects and intervals.
- Running many analyses and presenting only the smallest p-value.
- Switching to a one-sided test after seeing the direction.
- Confusing association with causation.
- Treating statistical significance as a decision rule without costs and practical thresholds.


## Worked example and interpretation

### What the helper actually does

The final notebook uses a short rule-based `recommend_test` helper to map four dataset examples to *starting* method families. It identifies ANES as an independent continuous-group comparison (Welch or Mann–Whitney), Grunfeld as paired continuous observations (paired t or signed-rank), Spector as a sparse independent categorical table (Fisher), and heart-transplant data as time-to-event observations (log-rank or survival regression). The row counts—944, 220, 32, and 69—are dataset sizes, **not** always counts of independent units. Grunfeld's 220 rows, for example, belong to 11 repeated firms.

### What still requires judgment

This helper cannot infer the actual estimand, missing-data mechanism, causal design, distribution shape, multiplicity family, censoring assumptions, or practical threshold from a few input flags. Its answer is therefore a prompt to inspect the relevant lesson, not an automated statistical recommendation. A complete report names the data source and independent unit; gives the effect in natural units with an interval; states the chosen method and assumptions; explains statistical and practical uncertainty; and describes what the analysis cannot establish. The final knowledge check asks the learner to defend those choices rather than merely recall a test name.

## Final knowledge check

1. Why does a 95% confidence interval that includes zero not prove zero effect?
2. What changes when observations are paired?
3. When is Fisher exact preferable to chi-square?
4. Why should effect sizes be reported in natural units?
5. How do Holm and Benjamini–Hochberg control different error criteria?

<details><summary>Answer guide</summary>

1. The interval contains a range of effects compatible with the data; zero is only one value.
2. Inference must use within-pair information and cannot assume row-level independence.
3. For sparse 2x2 tables or small expected counts where the chi-square approximation is unreliable.
4. Natural units connect uncertainty to scientific, operational, or clinical decisions.
5. Holm controls the chance of any false rejection in a family; BH controls the expected false discovery
   proportion among rejections under its assumptions.
</details>

## Where to go next

Study generalized linear models, hierarchical models, causal inference, sequential testing, missing-data
methods, robust statistics, and Bayesian modeling. The central habit remains the same: align the question,
design, estimand, model, uncertainty statement, and decision.


## Reproducibility checklist

- State the population, outcome, groups, and sampling unit.
- State $H_0$, $H_1$, alpha, and whether the test is one- or two-sided before looking at results.
- Verify independence from the design; use plots and diagnostics for distributional assumptions.
- Report the estimate, confidence interval, effect size, test statistic, degrees of freedom, and exact p-value.
- Separate statistical evidence from practical importance and acknowledge design limitations.
